<p style='text-align: center;'>

## <p style='text-align: center;'> **This is the Notebook for the CIFO Project**

### <p style='text-align: center;'> **Group: C37 Vestigial**




In [4]:
# Numerical arrays and vectorized operations used for triangle genomes and RMSE calculations.
import numpy as np

# DataFrame structure used to store and analyze results across multiple runs and configurations.
import pandas as pd

# Plotting library used to show final images and convergence curves inside the notebook.
import matplotlib.pyplot as plt

# Random utilities used by the genetic algorithm: initialization, selection, and mutation.
from random import randint, choices, sample, random

# OpenCV is used to load images, draw filled triangles, and save generated outputs.
import cv2

# ABC/abstractmethod are used to define a generic optimization solution template.
from abc import ABC, abstractmethod

# deepcopy prevents offspring/elites from accidentally sharing the same object in memory as parents.
from copy import deepcopy

# OS is used for file path manipulations and directory creation.
import os

# Time is used to measure the runtime of the algorithm and to timestamp output files.
import time

In [ ]:
class Solution(ABC):
    """
    Abstract base class defining the foundational blueprint for candidate solutions in optimization algorithms.

    This generic class establishes the required architecture for any evolutionary or single-point 
    optimization state. It mandates that all inheriting subclasses (such as `VermeerSolution`) 
    must implement their own specific data structure representation, random initialization logic, 
    and fitness value in order to be able to compare solutions and participate in the optimization process.
    """

    def __init__(self, repr=None):
        """
        Initializes the generic solution instance and establishes its genetic representation.

        If a specific representation is not passed during instantiation, this method automatically 
        invokes the abstract `random_initial_representation()` method to initialize a random starting state.

        Internal Variables:
            self.repr (Any): The core data structure (genome/state) of the solution. The exact data 
                type (e.g., numpy.ndarray, list, bitstring) depends on the specific subclass implementation.

        Parameters:
            repr (Any | None): An optional pre-existing representation to clone or inject. If None, 
                a completely randomized state is generated.
        """
        # If no representation is provided, create a random one using the subclass method.
        if repr is None:
            repr = self.random_initial_representation()

        # Store the representation/genome of the solution.
        self.repr = repr

    def __repr__(self):
        """
        Provides a string output of the solution's internal representation.

        Returns:
            str: The string-cast output of the `self.repr` data structure.
        """
        # Defines how the solution is displayed when printed.
        return str(self.repr)

    @abstractmethod
    def fitness(self):
        # Every specific solution must define how quality/fitness is calculated.
        pass

    @abstractmethod
    def random_initial_representation(self):
        # Every specific solution must define how a random starting solution is created.
        pass

In [ ]:
class VermeerSolution(Solution):
    """
    Represents a candidate painting composed of 100 semi-transparent colored triangles.

    This class handles the genetic representation (genome) of a single painting, 
    the logic to randomly initialize its 900 variables, the OpenCV rendering pipeline 
    to convert the genome into a 2D image matrix, and the Root Mean Square Error (RMSE) fitness evaluation.
    """

    def __init__(self, target_image, repr=None):
        """
        Initializes a candidate solution and establishes the geometric constraints of the canvas.

        Internal Variables:
            self.target_image (numpy.ndarray): The reference image array used later for RMSE calculation.
            self.width (int): Hardcoded canvas width restriction (300 pixels).
            self.height (int): Hardcoded canvas height restriction (400 pixels).
            self.num_triangles (int): Hardcoded limit defining the genome size (100 triangles).

        Parameters:
            target_image (numpy.ndarray): The target image matrix (BGR) the solution attempts to replicate.
            repr (numpy.ndarray | None): An optional pre-existing 100 x 9 genome array. If None, 
                the superclass will trigger `random_initial_representation()` to build a new one.
        """
        # Store the original image that the generated triangle image will be compared against.
        self.target_image = target_image

        # Project canvas size: 300x400 pixels.
        self.width = 300
        self.height = 400

        # Project requirement: use 100 triangles.
        self.num_triangles = 100

        # Calls Solution.__init__, which either stores a provided repr or creates a random one.
        super().__init__(repr=repr)

    def random_initial_representation(self):
        """
        Generates a completely randomized starting genome of 100 triangles.

        To prevent massive, screen-covering triangles in generation 0, this method uses a 
        bounding radius. It picks a random center point and generates three vertices tightly 
        grouped around that center, ensuring finer details can be evolved later.

        Internal Variables:
            random_repr (numpy.ndarray): A (100, 9) integer matrix initialized to zeros. 
                Each row holds [x1, y1, x2, y2, x3, y3, r, g, b].
            cx, cy (int): Randomly generated origin coordinates for a single triangle.
            radius (int): A hardcoded pixel bound (40) restricting how far vertices can spawn from the center.
            x1, y1, x2, y2, x3, y3 (int): Random coordinate pairs generated within the radius and clipped to the canvas.
            r, g, b (int): Randomly generated color channels between 0 and 255.

        Returns:
            random_repr (numpy.ndarray): The populated 100x9 integer matrix representing the new genome.
        """

        # Matrix with one row per triangle and 9 values per triangle.
        random_repr = np.zeros((self.num_triangles, 9), dtype=int)

        for i in range(self.num_triangles):
            # Pick a random center points in the canvas for the triangle
            cx = randint(0, self.width)
            cy = randint(0, self.height)

            # Define an max radius for the triangles
            radius = 40
            # Generate 3 random x-coordinates and 3 random y-coordinates within the range of the defined radius (clip helps to ensure 
            # vertices don't go outside the canvas). cx and cy serve as the central point, while the radius controls how far the vertices can be from that center.
            # This allows to control the size of the randomly intialized traingles (can start with small triangles)
            x1 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y1 = np.clip(cy + randint(-radius, radius), 0, self.height)
            x2 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y2 = np.clip(cy + randint(-radius, radius), 0, self.height)
            x3 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y3 = np.clip(cy + randint(-radius, radius), 0, self.height)
            # Generate one random RGB color for the whole triangle.
            r, g, b = [randint(0, 255) for _ in range(3)]
            # Store the triangle's vertices and color in the genome.
            random_repr[i] = [x1, y1, x2, y2, x3, y3, r, g, b]

        return random_repr

    def render_canvas(self):
        """
        Parses the numerical genome and paints it onto a 2D pixel matrix using OpenCV.

        The method iterates row-by-row through the genome. Because the genome dictates the 
        drawing sequence (Z-index), shapes at the end of the array will be drawn on top of 
        shapes at the beginning.

        Internal Variables:
            canvas (numpy.ndarray): A (400, 300, 3) matrix of unsigned 8-bit integers initialized to pure black.
            gene (numpy.ndarray): A 1D slice representing a single row (triangle) from `self.repr`.
            pts (numpy.ndarray): A 3x2 matrix of the triangle's integer coordinates, reshaped to (-1, 1, 2) 
                to comply with the specific contour formatting required by cv2.fillPoly.
            color (tuple): A tuple of three integers (R, G, B) extracted from the gene.

        Returns:
            canvas (numpy.ndarray): The finalized 3D pixel matrix representing the visually rendered painting.
        """

        # Start from a black image with shape: height x width x RGB channels.
        canvas = np.zeros((self.height, self.width, 3), dtype=np.uint8)

        for gene in self.repr:
            # Each gene is a row in the genome matrix, representing one triangle with 9 values: [x1, y1, x2, y2, x3, y3, r, g, b].
            # Extract the three triangle vertices in OpenCV's expected format.
            pts = np.array(
                [[gene[0], gene[1]], [gene[2], gene[3]], [gene[4], gene[5]]], np.int32
            )

            # Reshape points so cv2.fillPoly can draw the triangle.
            # Reshapes the 3x2 matrix of vertices into a format that OpenCV's fillPoly function expects, which is a list of contours shape (3, 1, 2).
            # 3 vertices, 1 contour, 2 coordinates (x and y).
            pts = pts.reshape((-1, 1, 2))

            # Extract the triangle color.
            # The loaded OpenCV image is BGR, reverse of the typical RGB order in the genome, 
            # but since we are only comparing pixel values for RMSE, the order does not affect the fitness calculation.
            color = (int(gene[6]), int(gene[7]), int(gene[8]))

            # Draw a filled triangle on the canvas.
            # cv2.fillPoly takes the canvas, a list of contours and the color to fill it with.
            cv2.fillPoly(canvas, [pts], color)

        return canvas

    def fitness(self):
        """
        Calculates the Root Mean Squared Error (RMSE) between the candidate painting and the target image.

        To prevent memory overflow/underflow artifacts that occur when subtracting unsigned 8-bit 
        integers (uint8) bounds 0 and 255, the method mathematically upcasts the image matrices to 32-bit floats before 
        performing the pixel-wise subtraction (allowing for negative values).

        Internal Variables:
            generated_image (numpy.ndarray): The 3D pixel matrix output yielded by `self.render_canvas()`.
            target_float, gen_float (numpy.ndarray): The respective image matrices cast to `np.float32`.
            diff (numpy.ndarray): The raw pixel-by-pixel arithmetic difference between the two float matrices.
            sq_diff (numpy.ndarray): The `diff` matrix mathematically squared to eliminate negative values.
            mse (float): The Mean Squared Error, calculated as the statistical mean of `sq_diff`.
            rmse (float): The final Root Mean Squared Error, calculated as the square root of the MSE.

        Returns:
            rmse (float): The calculated RMSE. Minimization problem, so solution with the lowest/lower RMSE is considered the best/better.
        """
        # Render this individual's genome into an image.
        generated_image = self.render_canvas()

        # Convert to float before subtracting, avoiding uint8 overflow/underflow errors.
        # Since limit of int8 is 0-255, subtracting two uint8 can wrap around (e.g., 10 - 20 would yield 246 instead of -10).
        target_float = self.target_image.astype(np.float32)
        gen_float = generated_image.astype(np.float32)

        # Pixel-by-pixel difference between target and generated image.
        diff = target_float - gen_float
        # RMSE = sqrt(mean(square(error))). Lower RMSE means a better image.
        sq_diff = np.square(diff)
        mse = np.mean(sq_diff)
        rmse = np.sqrt(mse)

        # The final fitness is the visual error (pixel to pixel) between the generated image and the target image.
        return rmse

In [9]:
# Path to local copies of the target image.
joao_path = r"C:\Users\joaoa\Desktop\CIFO\data\girl_pearl_earing.png"
goncalo_path = r"C:\CIfO\CIFO_Group37_Vestigial\Project\girl_pearl_earing.png"
rafa_path = r"Project\girl_pearl_earing.png"
gui_path = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\girl_pearl_earing.png"
# Load the original image using OpenCV.
original_image = cv2.imread(goncalo_path)

# Fail early with a clear error if the path is wrong or the image cannot be read.
if original_image is None:
    raise FileNotFoundError(f"OpenCV could not find the image at: {goncalo_path}")

# Create one random triangle painting to test the VermeerSolution class.
first_painting = VermeerSolution(target_image=original_image)

# Print its genome and RMSE fitness to confirm the representation and fitness function work.
print(f"{first_painting.repr} has fitness {first_painting.fitness()}")

[[ 66 194  93 239  62 212 154 170 172]
 [ 12  32   8   2  27  21 153  31 123]
 [ 93 257  51 246  19 277  38  45 106]
 [150  63  95  52 157  20 160 254  99]
 [258 393 300 353 300 400  18  55 111]
 [286  75 248 109 300 106  98 182 111]
 [  0 282   0 330  27 283  79 244  93]
 [258 366 255 400 300 383  57 116  25]
 [237 400 253 394 223 386  97  89  93]
 [245  52 247  63 273  69 242 108  58]
 [248 225 231 222 234 205 186  18 229]
 [ 78   0  45  30  47   0  66 107 219]
 [ 72 177  51 256 101 218 151  51  58]
 [216 368 266 395 230 336 238  81 213]
 [239  99 253 102 289 135 168 190  25]
 [294 332 300 313 300 367 247 208  15]
 [225 370 218 332 247 349 212 100 154]
 [160 340 133 340  87 394  88  45 175]
 [ 43 268  32 288  57 317 251  42 248]
 [132  10 114   0 129  56 186 139  51]
 [236 398 263 400 260 400   1  84 128]
 [251  82 229 124 274 104  99 140 162]
 [166 365 224 339 207 349 241  44 124]
 [  0 284  13 267  24 235 143  29 234]
 [252  57 254 132 210 108   9  88 109]
 [ 57  62  52  23   3   0

#### Vemeer 3

In [ ]:
class VermeerGA3:
    """
    Executes a highly parameterized Genetic Algorithm to approximate a target image using semi-transparent triangles.

    The class orchestrates selection, crossover, mutation, and dynamic fitness sharing across 
    a population of solutions. It supports ablation studies by allowing genetic operators to 
    be toggled via string identifiers.
    
    Sets up hyperparameters, dynamic dispatchers for genetic operators, and 
    generates the initial randomized population of VermeerSolutions.

    VermeerGA3 Class Comprises the following Parameters:

        Parameters:
            target_image (ndarray): The target image array to approximate.
            pop_size (int): The number of individuals in the population.
            generations (int): The total number of evolutionary generations to run.
            pc (float): The probability of crossover occurring between two parents.
            pm (float): The probability of mutation occurring for an individual.
            elitism_size (int): The number of top-performing individuals guaranteed to survive.
            tournament_size (int): The number of individuals sampled for tournament selection.
            selection_method (str): Identifier for the selection operator ("ranking" or "tournament").
            crossover_method (str): Identifier for the crossover operator ("cycle", "uniform", "two_point").
            mutation_method (str): Identifier for the Z-index mutation operator ("scramble", "inversion").
            use_crossover (bool): Flag to enable or disable the crossover phase.
            use_mutation (bool): Flag to enable or disable the mutation phase.
            use_fitness_sharing (bool): Initial flag state for dynamic fitness sharing.
            variance_threshold (float): The genotypic variance level that triggers fitness sharing.
            save_every (int): Generation interval at which to save the best rendering.
            output_dir (str): Directory path to save the generated images.
    """

    def __init__(
        self,
        target_image,
        pop_size=150,
        generations=2000,
        pc=0.85,
        pm=0.03,
        elitism_size=10,
        tournament_size=4,
        selection_method="ranking",  # "tournament" or "ranking"
        crossover_method="cycle",  # "cycle" or "two_point" or "uniform"
        mutation_method="swap",  # "scramble" or "inversion"
        use_crossover=True,
        use_mutation=True,
        use_fitness_sharing=False,
        variance_threshold=450,
        save_every=None,
        output_dir=None,
    ):

        self.target_image = target_image
        self.pop_size = pop_size
        self.generations = generations
        self.pc = pc
        self.pm = pm
        self.elitism_size = elitism_size

        # Genetic Operators
        self.selection_method = selection_method
        self.tournament_size = tournament_size
        self.crossover_method = crossover_method
        self.mutation_method = mutation_method

        self.use_crossover = use_crossover
        self.use_mutation = use_mutation
        self.use_fitness_sharing = use_fitness_sharing
        self.variance_threshold = variance_threshold
        self.save_every = save_every
        self.output_dir = output_dir

        # Initial random population
        self.population = [
            VermeerSolution(target_image=self.target_image)
            for _ in range(self.pop_size)
        ]

    #  Fitness sharing with Manhattan distance

    def apply_fitness_sharing(self):
        """
        Applies a mathematical penalty to genetically crowded individuals to handle premature convergence and improve diversity.

        Internal Variables:
            radius_share (float): The niche radius threshold (set to 0.15). Individuals within this normalized genetic distance penalize each other.
            alpha (float): The penalty shape factor (set to 1.0 for a linear penalty curve).
            flat_reprs (numpy.ndarray): A 2D matrix where each row is the flattened 900-variable genome of an individual.
            dist_matrix (numpy.ndarray): A 2D pairwise matrix holding the raw Manhattan (L1) distances between all individuals.
            norm_dist_matrix (numpy.ndarray): The dist_matrix dynamically scaled to a strictly [0.0, 1.0] range.
            similarity_matrix (numpy.ndarray): A matrix calculating the sharing function value based on radius_share and alpha.
            niche_counts (numpy.ndarray): A 1D array containing the sum of similarities for each individual, acting as a density multiplier.

        Returns:
            None: Modifies the `shared_fitness` attribute of each individual in `self.population` in place.
        """
        radius_share = 0.15 # Dictates the normalized genetic distance threshold for sharing. Individuals with a normalized distance below this value will penalize each other's fitness.
        alpha = 1.0 # The penalty shape factor. With alpha=1.0, the sharing function decreases linearly from 1 to 0 as distance goes from 0 to radius_share. Higher alpha values create a steeper drop-off, while lower values create a gentler slope.

        flat_reprs = np.vstack([ind.repr.flatten() for ind in self.population])

        # Manhattan distance between all pairs
        diff = flat_reprs[:, np.newaxis, :] - flat_reprs[np.newaxis, :, :]
        dist_matrix = np.sum(np.abs(diff), axis=-1)

        # Normalize distances to [0, 1]
        min_dist = np.min(dist_matrix)
        max_dist = np.max(dist_matrix)
        if max_dist == min_dist:
            norm_dist_matrix = np.zeros_like(dist_matrix)
        else:
            norm_dist_matrix = (dist_matrix - min_dist) / (max_dist - min_dist)

        # Similarity from distance
        similarity_matrix = np.where(
            norm_dist_matrix < radius_share,
            1.0 - (norm_dist_matrix / radius_share) ** alpha,
            0.0,
        )

        niche_counts = np.sum(similarity_matrix, axis=1)
        # Calculate shared fitness by multiplying the original fitness score by the niche count. 
        # Individuals in crowded niches (high niche count) will have their fitness reduced more significantly, 
        # while those in sparse niches will retain more of their original fitness.
        for idx, ind in enumerate(self.population):
            ind.shared_fitness = ind.fitness_score * niche_counts[idx]

    def evaluate_population(self):
        """
        Computes fitness scores and applies sorting logic to the current population.

        The method iterates through `self.population` and calculates `ind.fitness_score` (RMSE) 
        for any un-evaluated individual. It then conditionally sorts the population array based 
        on whether the fitness sharing parameter is active.

        Returns:
            None: Sorts `self.population` in place (ascending order, where index 0 is the best).
        """
        for ind in self.population:
            if not hasattr(ind, "fitness_score"):
                ind.fitness_score = ind.fitness() # Calculate fitness for any individual that hasn't been evaluated yet.

        # Sort the population based on shared fitness if sharing is enabled, otherwise sort by raw fitness score.
        if self.use_fitness_sharing:
            self.apply_fitness_sharing()
            self.population.sort(key=lambda ind: ind.shared_fitness)
        else:
            self.population.sort(key=lambda ind: ind.fitness_score)

    # Selection Methods 
    def select_parent(self):
        """
        Router method that selects selection operator based on class initialization parameters (ranking or tournament).

        Returns:
            VermeerSolution: A single selected parent individual.
        """
        if self.selection_method == "ranking":
            return self.ranking_selection()
        return self.tournament_selection()

    # Tournament selection

    def tournament_selection(self):
        """
        Executes a tournament selection protocol to choose a parent.

        Internal Variables:
            tournament (list): A random subset of `self.population` sized according to `self.tournament_size`.
            winner (VermeerSolution): The individual from the tournament list with the lowest applicable fitness score.

        Returns:
            winner (VermeerSolution): The winning individual.
        """

        # Randomly select tournament_size candidates.
        tournament = sample(self.population, self.tournament_size)

        if self.use_fitness_sharing:
            winner = sorted(tournament, key=lambda ind: ind.shared_fitness)[0]
        else:
            # The candidate with lowest RMSE wins the tournament.
            winner = sorted(tournament, key=lambda ind: ind.fitness_score)[0]
        return winner

    # Ranking selection

    def ranking_selection(self):
        """
        Executes linear ranking selection to choose a parent based on relative standing.
        Ex: In a a population of 5 individuals, the probabilities of selection would be 
        [0.33, 0.27, 0.20, 0.13, 0.07] for ranks 1 (best fitness) to 5 (worse fitness)respectively. 
        Where 0.33 = 5/15, 0.27 = 4/15, 0.20 = 3/15, 0.13 = 2/15, 0.07 = 1/15.

        Internal Variables:
            n (int): The total size of the population.
            ranks (numpy.ndarray): An array of integers descending from `n` to 1, assigning the highest weight to index 0.
            probs (numpy.ndarray): The normalized probability distribution array summing to 1.0.
            idx (int): The randomly chosen index based on the weighted probability array.

        Returns:
            VermeerSolution: The randomly selected parent individual.
        """
        n = len(self.population)
        ranks = np.arange(n, 0, -1)  # [n, n-1, ..., 1]
        probs = ranks / ranks.sum()
        idx = np.random.choice(n, p=probs)
        return self.population[idx]

    #  Crossover Methods 

    def crossover(self, parent1, parent2):
        """
        Router method that dispatches reproduction logic based on class initialization parameters.

        Parameters:
            parent1 (VermeerSolution): The first selected parent.
            parent2 (VermeerSolution): The second selected parent.

        Returns:
            tuple: A pair of crossed-over child individuals (child1, child2).
        """
        if self.crossover_method == "two_point":
            return self.two_point_crossover(parent1, parent2)
        elif self.crossover_method == "cycle":
            return self.cycle_crossover(parent1, parent2)
        # Default to Uniform
        return self.uniform_crossover(parent1, parent2)

    def uniform_crossover(self, parent1, parent2):
        """
        Executes uniform crossover by randomly selecting genes from either parent on a per-triangle basis.

        Internal Variables:
            child1, child2 (VermeerSolution): Instantiated shell objects to hold the new genetics.
            mask (numpy.ndarray): A binary matrix of 0s and 1s denoting inheritance source. 
                A value of 1 assigns the gene to child1 from parent1.

        Parameters:
            parent1 (VermeerSolution): The first selected parent.
            parent2 (VermeerSolution): The second selected parent.

        Returns:
            tuple: A tuple containing (child1, child2).
        """
        # Create a child object (random_representation of a solution(child)); then replace its genome using parent genes.
        child1 = VermeerSolution(
            target_image=self.target_image,
        )
        child2 = VermeerSolution(
            target_image=self.target_image,
        )
        # Mask has one 0/1 value per triangle.
        # 1 means take triangle from parent1; 0 means take triangle from parent2.
        mask = np.random.randint(0, 2, size=(child1.num_triangles, 1))
        # np.where applies the mask triangle by triangle.
        child1.repr = np.where(mask, parent1.repr, parent2.repr).copy()
        child2.repr = np.where(mask, parent2.repr, parent1.repr).copy()
        
        return child1, child2

    def cycle_crossover(self, parent1, parent2):
        """
        Executes a block-based uniform crossover (an adaptation of cycle crossover).

        Internal Variables:
            positions (list): A randomly shuffled list of index positions representing the genome sequence.
            cycle_len (int): A randomly generated length (bounded by n/4) determining block size.
            block (list): A sliced subset of `positions` that will be inherited sequentially from a single parent.
            new_repr1, new_repr2 (numpy.ndarray): Empty arrays gradually filled by alternating parent sources per block.

        Parameters:
            parent1 (VermeerSolution): The first selected parent.
            parent2 (VermeerSolution): The second selected parent.

        Returns:
            tuple: A tuple containing (child1, child2).
        """
        # Create child shells to hold the new genetics
        child1 = VermeerSolution(target_image=self.target_image)
        child2 = VermeerSolution(target_image=self.target_image)
        n = child1.num_triangles

        # Create a list of index positions and shuffle it to randomize the z-index inheritance pattern.
        positions = list(range(n))
        np.random.shuffle(positions)
        # Create empty arrays to hold the new representations as we fill them in blocks.
        new_repr1 = np.empty_like(parent1.repr)
        new_repr2 = np.empty_like(parent1.repr)
        
        i = 0 # Pointer to track our position in the shuffled index list.
        use_p1 = True # Flag to alternate between parent1 and parent2 as the source for each block. Starts with parent1.
        while i < n:
            # Randomly determine the length of the next block, ensuring it doesn't exceed n/4.
            cycle_len = randint(1, max(1, n // 4))
            # Define the block of indices that will be inherited from the current parent source.
            block = positions[i : i + cycle_len]
            
            source1 = parent1 if use_p1 else parent2
            source2 = parent2 if use_p1 else parent1
            
            for pos in block:
                new_repr1[pos] = source1.repr[pos]
                new_repr2[pos] = source2.repr[pos]
                
            use_p1 = not use_p1
            # Move the pointer forward by the length of the block.
            i += cycle_len
        # After filling the new representations in blocks, assign them to the child objects.
        child1.repr = new_repr1.copy()
        child2.repr = new_repr2.copy()
        
        return child1, child2

    def two_point_crossover(self, parent1, parent2):
        """
        Executes contiguous segment crossover (an adaptation of Order Crossover / OX).

        Internal Variables:
            c1, c2 (int): Two randomly selected integers representing the bounding start and end indices of the segment.
            child1.repr (numpy.ndarray): Initialized entirely with parent2 data, then mathematically overwritten 
                between indices `c1` and `c2` with parent1 data.

        Parameters:
            parent1 (VermeerSolution): The first selected parent.
            parent2 (VermeerSolution): The second selected parent.

        Returns:
            tuple: A tuple containing (child1, child2).
        """
        # Create child shells to hold the new genetics
        child1 = VermeerSolution(target_image=self.target_image)
        child2 = VermeerSolution(target_image=self.target_image)
        n = child1.num_triangles
        # Randomly select two crossover points and sort them to ensure c1 < c2. 
        # This defines the segment of the genome that will be swapped between parents.
        c1, c2 = sorted(np.random.choice(n, size=2, replace=False))

        child1.repr = parent2.repr.copy()
        child1.repr[c1:c2] = parent1.repr[c1:c2]

        child2.repr = parent1.repr.copy()
        child2.repr[c1:c2] = parent2.repr[c1:c2]
        
        return child1, child2

    #  Mutation 

    def mutate(self, child):
        """
        Applies physical coordinate/color micro-mutations and broad Z-index structural mutations.

        Internal Variables:
            mutation_type (float): A randomly generated float between 0.0 and 1.0 determining if a 
                triangle undergoes a micro-nudge (< 0.90) or a complete reset (> 0.90).
            The method subsequently routes to `scramble_mutation` or `inversion_mutation` based on class parameters.
            `child.fitness_score` is explicitly deleted to flag the individual for re-evaluation in the next cycle.

        Parameters:
            child (VermeerSolution): The individual solution targeted for mutation.

        Returns:
            child (VermeerSolution): The mutated child individual.
        """
        # 1. Standard Nudge/Replace Mutation (Crucial for exploring shapes/colors!)
        for i in range(child.num_triangles):
            if random() < self.pm:
                mutation_type = random()
                if mutation_type < 0.90:
                    # 90% of mutations are small nudges to preserve useful structures.
                    child.repr[i][0:6] += np.random.randint(-10, 11, 6)  # coordinates shifted by up to 10 pixels in any direction
                    child.repr[i][6:9] += np.random.randint(-15, 16, 3)  # color shifted by up to 15 in any channel

                    # Keep x-coordinates inside the image width.
                    child.repr[i][0] = np.clip(child.repr[i][0], 0, child.width)
                    child.repr[i][2] = np.clip(child.repr[i][2], 0, child.width)
                    child.repr[i][4] = np.clip(child.repr[i][4], 0, child.width)

                    # Keep y-coordinates inside the image height.
                    child.repr[i][1] = np.clip(child.repr[i][1], 0, child.height)
                    child.repr[i][3] = np.clip(child.repr[i][3], 0, child.height)
                    child.repr[i][5] = np.clip(child.repr[i][5], 0, child.height)

                    # Keep color channels valid.
                    child.repr[i][6:9] = np.clip(child.repr[i][6:9], 0, 255)
                    pass
                else:
                    # Full triangle replace
                    child.repr[i] = [
                        randint(0, child.width),
                        randint(0, child.height),
                        randint(0, child.width),
                        randint(0, child.height),
                        randint(0, child.width),
                        randint(0, child.height),
                        randint(0, 255),
                        randint(0, 255),
                        randint(0, 255),
                    ]
                    pass
        if random() < self.pm:
            if self.mutation_method == "scramble":
                child = self.scramble_mutation(child)
            elif self.mutation_method == "inversion":
                child = self.inversion_mutation(child)

        if hasattr(child, "fitness_score"):
            del child.fitness_score
        return child

    def scramble_mutation(self, child):
        """
        Shuffles the physical Z-index rendering order of a subset of triangles.

        Internal Variables:
            c1, c2 (int): Two randomly selected bounding indices.
            segment (numpy.ndarray): A sub-slice of the child's representation matrix. 
                `np.random.shuffle` is applied to reorder the rows strictly within this segment.

        Parameters:
            child (VermeerSolution): The individual to undergo Z-index modification.

        Returns:
            child (VermeerSolution): The mutated child individual.
        """
        n = child.num_triangles
        c1, c2 = sorted(np.random.choice(n, size=2, replace=False))
        segment = child.repr[c1:c2].copy()
        np.random.shuffle(segment)
        child.repr[c1:c2] = segment
        return child

    def inversion_mutation(self, child):
        """
        Reverses the physical Z-index rendering order of a subset of triangles. 
        Similar to scramble but instead of random shuffling, the order is flipped, helping preseve structural integrity 
        (valuable structural patterns can still be maintained).

        Internal Variables:
            c1, c2 (int): Two randomly selected bounding indices. 
                The array slice `child.repr[c1:c2]` is mathematically reversed using `[::-1]`.

        Parameters:
            child (VermeerSolution): The individual to undergo Z-index modification.

        Returns:
            child (VermeerSolution): The mutated child individual.
        """
        n = child.num_triangles
        c1, c2 = sorted(np.random.choice(n, size=2, replace=False))
        child.repr[c1:c2] = child.repr[c1:c2][::-1]
        return child

    #  Diversity tracking 

    def calculate_population_variance(self):
        """
        Calculates the aggregate genotypic diversity across the entire population space.

        Internal Variables:
            flat_reprs (numpy.ndarray): A 2D array generated by `np.vstack` containing flattened copies of all individual genomes.
            gene_variances (numpy.ndarray): An array holding the calculated variance per column/gene across the population grid.

        Returns:
            float: The statistical mean of the calculated `gene_variances`.
        """
        flat_reprs = np.vstack([ind.repr.flatten() for ind in self.population])
        gene_variances = np.var(flat_reprs, axis=0)
        return np.mean(gene_variances)

    def maybe_save_generation(self, gen):
        """
        Conditionally renders and saves the absolute best structural canvas to the disk.

        Internal Variables:
            image (numpy.ndarray): The BGR pixel array generated by calling `render_canvas()` on the leading individual.
            
        Parameters:
            gen (int): The current evolutionary generation number being processed.

        Returns:
            Writes a `.png` file of the best solution (drawn vermeer solution) to disk if constraints are met.
        """
        if self.save_every is None or self.output_dir is None:
            return
        if gen % self.save_every != 0:
            return

        os.makedirs(self.output_dir, exist_ok=True)
        image = self.population[0].render_canvas()
        cv2.imwrite(f"{self.output_dir}/gen_{gen}.png", image)

    # ---------- Main loop ----------

    def run(self):
        """
        Executes the primary evolutionary loop.

        Internal Variables:
            fitness_history (list): Accumulator tracking the `best_current_solution.fitness_score` per generation.
            variance_history (list): Accumulator tracking the `current_variance` per generation.
            best_current_solution (VermeerSolution): The absolute lowest RMSE individual in a given generation.
            elites (list): A statically sorted subset of the top `self.elitism_size` individuals preserved without modification.
            next_generation (list): The newly constructed population array filled via reproduction mechanics.

        Returns:
            tuple: A data structure containing:
                - best_final_solution (VermeerSolution): The globally best solution upon completion.
                - fitness_history (list): List of floats tracking historical RMSE progression.
                - variance_history (list): List of floats tracking historical genotypic variance.
        """
        print(f"Starting Evolution for {self.generations} generations...")

        fitness_history = []
        variance_history = []

        self.evaluate_population()

        for gen in range(self.generations):
            best_current_solution = min(
                self.population, key=lambda ind: ind.fitness_score
            )
            fitness_history.append(best_current_solution.fitness_score)

            if gen % 50 == 0 or gen == self.generations - 1:
                current_variance = self.calculate_population_variance()
                
            variance_history.append(current_variance)

            # Dynamic fitness sharing functions by triggering fitness sharing when the population variance drops below the defined threshold.
            if current_variance < self.variance_threshold:
                if not self.use_fitness_sharing:
                    print(
                        f"\n Variance dropped to {current_variance:.2f} "
                        f"Fitness Sharing Activated at Gen: {gen}"
                    )
                self.use_fitness_sharing = True
                self.apply_fitness_sharing()

            if gen % 50 == 0:
                print(
                    f"Generation {gen} | Best RMSE: "
                    f"{best_current_solution.fitness_score:.2f} | "
                    f"Variance: {current_variance:.2f}"
                )
                self.maybe_save_generation(gen)

            # Elitism (Performs elitism by copying the top `self.elitism_size` individuals directly into the next generation without modification.)
            elites = sorted(self.population, key=lambda ind: ind.fitness_score)[:self.elitism_size]

            next_generation = [
                deepcopy(ind) for ind in elites
            ]

            # Fill the rest with offspring
            while len(next_generation) < self.pop_size:
                p1 = self.select_parent()

                if self.use_crossover and random() < self.pc:
                    p2 = self.select_parent()
                    child1, child2 = self.crossover(p1, p2)

                    if self.use_mutation:
                        child1 = self.mutate(child1)
                        child2 = self.mutate(child2)

                    next_generation.append(child1)
                    if len(next_generation) < self.pop_size:
                        next_generation.append(child2)
                else:
                    child = deepcopy(p1)
                    if self.use_mutation:
                        child = self.mutate(child)
                    next_generation.append(child)

            self.population = next_generation
            self.evaluate_population()

        self.evaluate_population()
        
        best_final_solution = min(self.population, key=lambda ind: ind.fitness_score)
        print(
            f"Evolution Complete! Final RMSE: "
            f"{best_final_solution.fitness_score:.2f}"
        )
        return best_final_solution, fitness_history, variance_history

#### Test run of the final class VermeerGA3 

In [29]:
# Baseline Experimment Implementation
exp_name = "Control_Baseline"
notes = "Standard benchmark run (Tournament + Uniform + Scramble). Fitness Sharing turned off for natural variation testing."

# Algorithm Hyperparameters
selection = "tournament"
crossover = "uniform"
mutation = "scramble"
n_runs = 20

pop_size = 150
generations = 2000
pc = 0.85
pm = 0.03
elitism_size = 10
tournament_size = 4
variance_threshold = 550
use_fitness_sharing = False

print(f"\n[{time.strftime('%H:%M:%S')}] Starting batch: {exp_name} ({n_runs} Runs)")

variant_dir = os.path.join("data", "results", exp_name)
if not os.path.exists(variant_dir):
    os.makedirs(variant_dir)

all_runs_history = []
final_rmses = []

for run in range(1, n_runs + 1):
    print(f"Run {run}/{n_runs}...")

    run_dir = os.path.join(variant_dir, f"Run_{run}")
    if not os.path.exists(run_dir):
        os.makedirs(run_dir)

    ga = VermeerGA3(
        target_image=original_image,
        pop_size=pop_size,
        generations=generations,
        pc=pc,
        pm=pm,
        elitism_size=elitism_size,
        selection_method=selection,
        crossover_method=crossover,
        mutation_method=mutation,
        use_fitness_sharing=use_fitness_sharing,
        tournament_size=tournament_size, 
        variance_threshold=variance_threshold 
    )

    best_sol, fitness_history, variance_history = ga.run()

    # --- NOVO: Preparar o dicionário de histórico pedido ---
    history_dict = {}
    for gen in range(len(fitness_history)):
        history_dict[gen] = {
            "fitness": fitness_history[gen],
            "variance": (
                variance_history[gen]
                if gen < len(variance_history)
                else variance_history[-1]
            ),
        }

    # --- NOVO: Gravação detalhada de Hiperparâmetros ---
    hyperparameter_path = os.path.join(run_dir, "hyperparameters.txt")
    with open(hyperparameter_path, "w") as file:
        file.write(f"GA3 Hyperparameters for {exp_name} | Run Number: {run}\n\n")

        if notes:
            file.write(f"Notes for this run: {notes}\n\n")

        file.write(f"--- VARIANT SETUP ---\n")
        file.write(f"Selection Method: {selection}\n")
        file.write(f"Crossover Method: {crossover}\n")
        file.write(f"Mutation Method: {mutation}\n\n")

        file.write(f"--- BASE PARAMETERS ---\n")
        file.write(f"Population Size: {pop_size}\n")
        file.write(f"Generations: {generations}\n")
        file.write(f"Crossover Probability (pc): {pc}\n")
        file.write(f"Mutation Probability (pm): {pm}\n")
        file.write(f"Elitism Size: {elitism_size}\n")
        file.write(f"Tournament Size: {tournament_size}\n")
        file.write(f"Fitness Sharing: {use_fitness_sharing}\n")
        file.write(f"Variance Threshold for Fitness Sharing: {variance_threshold}\n\n")

        file.write(f"--- GENERATION HISTORY ---\n")
        for gen, metrics in history_dict.items():
            if gen % 50 == 0:
                fit = metrics["fitness"]
                var = metrics["variance"]
                file.write(f"Gen {gen} | Fitness: {fit:.2f} | Variance: {var:.2f}\n")

        file.write(f"\n--- FINAL RESULTS ---\n")
        file.write(f"Final RMSE Score: {best_sol.fitness_score:.2f}\n")
        file.write(f"Final Variance: {variance_history[-1]:.2f}\n")

    # Guardar a imagem e o gráfico
    cv2.imwrite(os.path.join(run_dir, "final_result.png"), best_sol.render_canvas())

    plt.figure(figsize=(8, 4))
    plt.plot(fitness_history, color="black", linewidth=1.5)
    plt.title(f"Convergence - {exp_name} Run {run} (Final: {fitness_history[-1]:.2f})")
    plt.xlabel("Generation")
    plt.ylabel("RMSE")
    plt.grid(True)
    plt.savefig(os.path.join(run_dir, "convergence_plot.png"), bbox_inches="tight")
    plt.close()

    all_runs_history.append(fitness_history)
    final_rmses.append(fitness_history[-1])

print(
    f"\n[{time.strftime('%H:%M:%S')}] Batch {exp_name} finished! Best final RMSE: {np.mean(final_rmses):.2f}"
)

# --- GUARDAR DADOS GLOBAIS EM CSV ---
history_matrix = np.array(all_runs_history)

raw_df = pd.DataFrame(history_matrix)
raw_df.index = [f"Run_{i+1}" for i in range(n_runs)]
raw_df.columns = [f"Gen_{i}" for i in range(generations + 1)]
raw_df.to_csv(os.path.join(variant_dir, "raw_fitness_data.csv"))

stats_df = pd.DataFrame(
    {
        "Generation": np.arange(generations + 1),
        "Mean_RMSE": np.mean(history_matrix, axis=0),
        "Std_Dev": np.std(history_matrix, axis=0),
        "Best_RMSE": np.min(history_matrix, axis=0),
    }
)
stats_df.to_csv(os.path.join(variant_dir, "statistical_summary.csv"), index=False)


[23:26:18] Starting batch: Control_Baseline (20 Runs)
Run 1/20...
Starting Evolution for 2000 generations...
Generation 0 | Best RMSE: 99.79 | Variance: 8953.39
Generation 50 | Best RMSE: 54.04 | Variance: 1667.84


KeyboardInterrupt: 

In [ ]:
# Genetic Algorithm Crossover Implementation
exp_name = "Crossover_Variant_TwoPoint"
notes = "Crossover Insulation: Testing with Two-Point (OX) to evaluate layer construction. Tournament Selection and Swap Mutation."

# Algorithm Hyperparameters
selection = "tournament"
crossover = "two_point"  # <-- Change Made
mutation = "scramble"
n_runs = 20

pop_size = 150
generations = 2000
pc = 0.85
pm = 0.03
elitism_size = 10
tournament_size = 4
variance_threshold = 550
use_fitness_sharing = False

print(f"\n[{time.strftime('%H:%M:%S')}] Starting batch: {exp_name} ({n_runs} Runs)")

variant_dir = os.path.join("data", "results", exp_name)
if not os.path.exists(variant_dir):
    os.makedirs(variant_dir)

all_runs_history = []
final_rmses = []

for run in range(1, n_runs + 1):
    print(f"Run {run}/{n_runs}...")

    run_dir = os.path.join(variant_dir, f"Run_{run}")
    if not os.path.exists(run_dir):
        os.makedirs(run_dir)

    ga = VermeerGA3(
        target_image=original_image,
        pop_size=pop_size,
        generations=generations,
        pc=pc,
        pm=pm,
        elitism_size=elitism_size,
        selection_method=selection,
        crossover_method=crossover,
        mutation_method=mutation,
        use_fitness_sharing=use_fitness_sharing,
        tournament_size=tournament_size, 
        variance_threshold=variance_threshold 
    )

    best_sol, fitness_history, variance_history = ga.run()

    # Preparar o dicionário de histórico
    history_dict = {}
    for gen in range(len(fitness_history)):
        history_dict[gen] = {
            "fitness": fitness_history[gen],
            "variance": (
                variance_history[gen]
                if gen < len(variance_history)
                else variance_history[-1]
            ),
        }

    # Gravação detalhada de Hiperparâmetros
    hyperparameter_path = os.path.join(run_dir, "hyperparameters.txt")
    with open(hyperparameter_path, "w") as file:
        file.write(f"GA3 Hyperparameters for {exp_name} | Run Number: {run}\n\n")

        if notes:
            file.write(f"Notes for this run: {notes}\n\n")

        file.write(f"--- VARIANT SETUP ---\n")
        file.write(f"Selection Method: {selection}\n")
        file.write(f"Crossover Method: {crossover}\n")
        file.write(f"Mutation Method: {mutation}\n\n")

        file.write(f"--- BASE PARAMETERS ---\n")
        file.write(f"Population Size: {pop_size}\n")
        file.write(f"Generations: {generations}\n")
        file.write(f"Crossover Probability (pc): {pc}\n")
        file.write(f"Mutation Probability (pm): {pm}\n")
        file.write(f"Elitism Size: {elitism_size}\n")
        file.write(f"Tournament Size: {tournament_size}\n")
        file.write(f"Fitness Sharing: {use_fitness_sharing}\n")
        file.write(f"Variance Threshold for Fitness Sharing: {variance_threshold}\n\n")

        file.write(f"--- GENERATION HISTORY ---\n")
        for gen, metrics in history_dict.items():
            if gen % 50 == 0:
                fit = metrics["fitness"]
                var = metrics["variance"]
                file.write(f"Gen {gen} | Fitness: {fit:.2f} | Variance: {var:.2f}\n")

        file.write(f"\n--- FINAL RESULTS ---\n")
        file.write(f"Final RMSE Score: {best_sol.fitness_score:.2f}\n")
        file.write(f"Final Variance: {variance_history[-1]:.2f}\n")

    # Guardar a imagem e o gráfico
    cv2.imwrite(os.path.join(run_dir, "final_result.png"), best_sol.render_canvas())

    plt.figure(figsize=(8, 4))
    plt.plot(
        fitness_history, color="green", linewidth=1.5
    )  # Cor Verde para o Two-Point
    plt.title(f"Convergence - {exp_name} Run {run} (Final: {fitness_history[-1]:.2f})")
    plt.xlabel("Generation")
    plt.ylabel("RMSE")
    plt.grid(True)
    plt.savefig(os.path.join(run_dir, "convergence_plot.png"), bbox_inches="tight")
    plt.close()

    all_runs_history.append(fitness_history)
    final_rmses.append(fitness_history[-1])

print(
    f"\n[{time.strftime('%H:%M:%S')}] Batch {exp_name} finished! Best final RMSE: {np.mean(final_rmses):.2f}"
)

# --- GUARDAR DADOS GLOBAIS EM CSV ---
history_matrix = np.array(all_runs_history)

raw_df = pd.DataFrame(history_matrix)
raw_df.index = [f"Run_{i+1}" for i in range(n_runs)]
raw_df.columns = [f"Gen_{i}" for i in range(generations + 1)]
raw_df.to_csv(os.path.join(variant_dir, "raw_fitness_data.csv"))

stats_df = pd.DataFrame(
    {
        "Generation": np.arange(generations + 1),
        "Mean_RMSE": np.mean(history_matrix, axis=0),
        "Std_Dev": np.std(history_matrix, axis=0),
        "Best_RMSE": np.min(history_matrix, axis=0),
    }
)
stats_df.to_csv(os.path.join(variant_dir, "statistical_summary.csv"), index=False)


[21:40:51] Starting batch: Crossover_Variant_TwoPoint (30 Runs)
Run 1/30...
Starting Evolution for 500 generations...
Generation 0 | Best RMSE: 98.50 | Variance: 8838.24


KeyboardInterrupt: 

In [ ]:
# Implementation Configuration for Mutation Variant: Inversion
exp_name = "Mutation_Variant_Inversion"
notes = "Mutation Isolation: Inversion test to evaluate drastic space exploration jumps. Tournament Selection and Crossover Uniform."

# Algorithm Hyperparameters
selection = "tournament"
crossover = "uniform"
mutation = "inversion"  # Change Made
n_runs = 20

pop_size = 150
generations = 2000
pc = 0.85
pm = 0.03
elitism_size = 10
tournament_size = 4
variance_threshold = 550
use_fitness_sharing = False

print(f"\n[{time.strftime('%H:%M:%S')}] Starting batch: {exp_name} ({n_runs} Runs)")

variant_dir = os.path.join("data", "results", exp_name)
if not os.path.exists(variant_dir):
    os.makedirs(variant_dir)

all_runs_history = []
final_rmses = []

for run in range(1, n_runs + 1):
    print(f"Run {run}/{n_runs}...")

    run_dir = os.path.join(variant_dir, f"Run_{run}")
    if not os.path.exists(run_dir):
        os.makedirs(run_dir)

    ga = VermeerGA3(
        target_image=original_image,
        pop_size=pop_size,
        generations=generations,
        pc=pc,
        pm=pm,
        elitism_size=elitism_size,
        selection_method=selection,
        crossover_method=crossover,
        mutation_method=mutation,
        use_fitness_sharing=use_fitness_sharing,
        tournament_size=tournament_size, 
        variance_threshold=variance_threshold 
    )

    best_sol, fitness_history, variance_history = ga.run()

    # Preparar o dicionário de histórico
    history_dict = {}
    for gen in range(len(fitness_history)):
        history_dict[gen] = {
            "fitness": fitness_history[gen],
            "variance": (
                variance_history[gen]
                if gen < len(variance_history)
                else variance_history[-1]
            ),
        }

    # Gravação detalhada de Hiperparâmetros
    hyperparameter_path = os.path.join(run_dir, "hyperparameters.txt")
    with open(hyperparameter_path, "w") as file:
        file.write(f"GA3 Hyperparameters for {exp_name} | Run Number: {run}\n\n")

        if notes:
            file.write(f"Notes for this run: {notes}\n\n")

        file.write(f"--- VARIANT SETUP ---\n")
        file.write(f"Selection Method: {selection}\n")
        file.write(f"Crossover Method: {crossover}\n")
        file.write(f"Mutation Method: {mutation}\n\n")

        file.write(f"--- BASE PARAMETERS ---\n")
        file.write(f"Population Size: {pop_size}\n")
        file.write(f"Generations: {generations}\n")
        file.write(f"Crossover Probability (pc): {pc}\n")
        file.write(f"Mutation Probability (pm): {pm}\n")
        file.write(f"Elitism Size: {elitism_size}\n")
        file.write(f"Tournament Size: {tournament_size}\n")
        file.write(f"Fitness Sharing: {use_fitness_sharing}\n")
        file.write(f"Variance Threshold for Fitness Sharing: {variance_threshold}\n\n")

        file.write(f"--- GENERATION HISTORY ---\n")
        for gen, metrics in history_dict.items():
            if gen % 50 == 0:
                fit = metrics["fitness"]
                var = metrics["variance"]
                file.write(f"Gen {gen} | Fitness: {fit:.2f} | Variance: {var:.2f}\n")

        file.write(f"\n--- FINAL RESULTS ---\n")
        file.write(f"Final RMSE Score: {best_sol.fitness_score:.2f}\n")
        file.write(f"Final Variance: {variance_history[-1]:.2f}\n")

    # Guardar a imagem e o gráfico
    cv2.imwrite(os.path.join(run_dir, "final_result.png"), best_sol.render_canvas())

    plt.figure(figsize=(8, 4))
    plt.plot(
        fitness_history, color="red", linewidth=1.5
    )  # Cor Vermelha para o Scramble
    plt.title(f"Convergence - {exp_name} Run {run} (Final: {fitness_history[-1]:.2f})")
    plt.xlabel("Generation")
    plt.ylabel("RMSE")
    plt.grid(True)
    plt.savefig(os.path.join(run_dir, "convergence_plot.png"), bbox_inches="tight")
    plt.close()

    all_runs_history.append(fitness_history)
    final_rmses.append(fitness_history[-1])

print(
    f"\n[{time.strftime('%H:%M:%S')}] Batch {exp_name} finished! Best final RMSE: {np.mean(final_rmses):.2f}"
)

# --- GUARDAR DADOS GLOBAIS EM CSV ---
history_matrix = np.array(all_runs_history)

raw_df = pd.DataFrame(history_matrix)   
raw_df.index = [f"Run_{i+1}" for i in range(n_runs)]
raw_df.columns = [f"Gen_{i}" for i in range(generations + 1)]
raw_df.to_csv(os.path.join(variant_dir, "raw_fitness_data.csv"))

stats_df = pd.DataFrame(
    {
        "Generation": np.arange(generations + 1),
        "Mean_RMSE": np.mean(history_matrix, axis=0),
        "Std_Dev": np.std(history_matrix, axis=0),
        "Best_RMSE": np.min(history_matrix, axis=0),
    }
)
stats_df.to_csv(os.path.join(variant_dir, "statistical_summary.csv"), index=False)


[23:48:06] Starting batch: Mutation_Variant_Inversion (20 Runs)
Run 1/20...
Starting Evolution for 2000 generations...
Generation 0 | Best RMSE: 98.49 | Variance: 8809.62
Generation 50 | Best RMSE: 55.20 | Variance: 914.61
Generation 100 | Best RMSE: 47.33 | Variance: 768.33

 Variance dropped to 323.69 Fitness Sharing Activated at Gen: 150
Generation 150 | Best RMSE: 44.01 | Variance: 323.69
Generation 200 | Best RMSE: 42.91 | Variance: 1861.92
Generation 250 | Best RMSE: 42.04 | Variance: 1936.55
Generation 300 | Best RMSE: 40.95 | Variance: 1906.20
Generation 350 | Best RMSE: 40.26 | Variance: 1959.01
Generation 400 | Best RMSE: 39.98 | Variance: 2064.35
Generation 450 | Best RMSE: 39.35 | Variance: 2034.06
Generation 500 | Best RMSE: 39.15 | Variance: 2148.68
Generation 550 | Best RMSE: 38.81 | Variance: 2089.16
Generation 600 | Best RMSE: 38.47 | Variance: 2232.43
Generation 650 | Best RMSE: 38.11 | Variance: 1920.77
Generation 700 | Best RMSE: 37.65 | Variance: 2101.80
Generatio

ValueError: Length mismatch: Expected axis has 2000 elements, new values have 2001 elements